# Lesson 14 · Electromagnetic induction

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/14_electromagnetic_induction.ipynb)

**For:** A-level Physics (for example AQA 3.7.5.3 to 3.7.5.5, OCR A 6.3.3, Edexcel Topic 7) and IB Physics (D.4 induction). Check your own specification for the exact wording.

**Before this lesson:** Lesson 1, *Simple harmonic motion* (angular frequency and sinusoidal graphs, which describe the emf of a rotating coil); Lesson 6, *Charged particles in electric and magnetic fields* (the magnetic force on moving charges and on a current-carrying wire). Also assumed: the field patterns of a bar magnet and of a current-carrying wire.

**Leads to:** the end of this route for now.

**You will be able to:**

1. calculate magnetic flux $\Phi = BA\cos\theta$ and flux linkage $N\Phi$, knowing which angle $\theta$ is
2. state Faraday's law, $\varepsilon = -\dfrac{d(N\Phi)}{dt}$, and use it for average and instantaneous emfs
3. use Lenz's law to find the direction of an induced current, and explain it as conservation of energy
4. derive $\varepsilon = BLv$ for a conductor moving through a field, and find the force needed to keep it moving
5. explain the sinusoidal emf of a rotating coil, $\varepsilon = BAN\omega\sin\omega t$, and use peak and r.m.s. values
6. explain how a changing current induces an emf in a nearby coil with no moving parts

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Before some experiments a **Predict** box asks you to commit to an answer; write it down, run the cell, then answer the **Explain** box below it. Answers to both are at the very end, after the worked solutions. Exercises are at the end, with worked solutions below them.

In [ ]:
# Install the course API in a fresh Colab runtime or upgrade an older installation.
import importlib.util
import subprocess
import sys
from importlib.metadata import version

if importlib.util.find_spec("qplay") is None or int(version("quantum-playground").split(".")[0]) < 1:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quantum-playground>=1,<2"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import mu_0      # permeability of free space, 4*pi*1e-7 T m/A (to 9 s.f.)
from qplay import em, integrators

## 1. Magnetic flux and flux linkage

Magnetic flux measures how much field passes **through** an area. For a flat loop of area $A$ in a uniform field $B$,

$$\Phi = BA\cos\theta ,$$

where $\theta$ is the angle between the field and the **normal** to the loop (the line at right angles to its plane). The flux is greatest, $BA$, when the field passes straight through, and zero when the loop is edge-on to the field. Its unit is the weber: $1\ \mathrm{Wb} = 1\ \mathrm{T\,m^2}$.

A coil of $N$ turns has the same flux through every turn, so we use the **flux linkage**

$$N\Phi = BAN\cos\theta \qquad (\text{unit: Wb-turns, or just Wb}).$$

**Exam tip:** if a question gives the angle between the field and the *plane* of the coil, that angle is $90^\circ - \theta$, so use $\sin$ instead of $\cos$.

In [ ]:
N, A, B = 100, 0.010, 0.20                 # turns, area in m^2 (10 cm x 10 cm), field in T
theta = np.linspace(0, 360, 361)
linkage = B * A * N * np.cos(np.radians(theta))

plt.figure(figsize=(7, 3.2))
plt.plot(theta, linkage)
for angle, where, offset in [(0, "field straight through", (8, -28)), (90, "edge-on", (8, -28)), (180, "flipped over", (8, 8))]:
    value = round(B * A * N * np.cos(np.radians(angle)), 12)   # round away the 1e-17 left by cos(90 degrees)
    plt.plot(angle, value, "o", color="C3")
    plt.annotate(f"{where}\nN$\\Phi$ = {value:.2f} Wb", (angle, value), textcoords="offset points", xytext=offset, fontsize=8)
plt.axhline(0, color="0.7", lw=0.6)
plt.xticks(range(0, 361, 45)); plt.xlabel(r"angle $\theta$ between B and the normal / degrees"); plt.ylabel(r"flux linkage $N\Phi$ / Wb")
plt.title("Flux linkage of a 100-turn, 10 cm square coil in 0.20 T");

Flipping the coil over makes the flux linkage negative: the field now enters through what was the back face. That sign matters as soon as the flux starts to **change**.

## 2. Faraday's law and Lenz's law

Michael Faraday found in 1831 that a steady field induces nothing, but a **changing** flux linkage induces an emf. In modern form, **Faraday's law** says

$$\varepsilon = -\frac{d(N\Phi)}{dt}, \qquad\text{or on average}\qquad \varepsilon = -\frac{\Delta(N\Phi)}{\Delta t}.$$

The induced emf equals the **rate of change** of flux linkage. It does not matter how the flux changes: by moving a magnet, moving or turning the coil, or changing the current in a nearby electromagnet.

The minus sign is **Lenz's law**: *the induced current flows in the direction that opposes the change that produced it*. Push the north pole of a magnet into a coil and the induced current makes the near end of the coil a north pole, which pushes back. Pull the magnet out and the near end becomes a south pole, which pulls back.

Why must it oppose? Suppose the induced current *helped* the change instead. Then a tiny push on the magnet would induce a current that pulled the magnet in faster, which would induce a bigger current, and so on, creating energy from nothing. Lenz's law is the principle of **conservation of energy**: the work you do against the opposing force is exactly the electrical energy the circuit receives. Section 3 checks this with numbers.

## 3. A rod moving on rails: $\varepsilon = BLv$

A metal rod of length $L$ slides at speed $v$ along two parallel rails, with a uniform field $B$ at right angles to the plane of the rails. The rod and the rails form a loop whose area grows by $Lv\,\Delta t$ in a time $\Delta t$. So the flux grows by $BLv\,\Delta t$, and Faraday's law gives

$$\varepsilon = \frac{\Delta\Phi}{\Delta t} = BLv .$$

(You can also see it from the force on the charges: $F = BQv$ pushes them along the rod until the electric field $E$ they build up balances it, $QE = BQv$, so $\varepsilon = EL = BLv$.)

The next cell moves a rod at constant speed, works out the flux through the loop at each moment, and differentiates it numerically, which is Faraday's law. It compares the result with `em.rod_emf`, which uses $BLv$.

Try it: compare the rod’s lost kinetic energy with heat in the rails in the [explorer](../explorer/#coil).

In [ ]:
B, L, v = 0.40, 0.15, 2.0                  # T, m, m/s
t = np.linspace(0, 0.5, 501)
x = 0.10 + v * t                           # rod position, so the loop has area L * x
flux = B * L * x
faraday = np.gradient(flux, t)             # d(Phi)/dt, step by step

print(f"Faraday, dPhi/dt from the changing area: {faraday.mean():.4f} V (spread {np.ptp(faraday):.1e} V)")
print(f"em.rod_emf, BLv:                         {em.rod_emf(B, L, v):.4f} V")

Now connect a resistor $R$ between the ends of the rails. A current $I = \varepsilon/R = BLv/R$ flows round the loop, and the rod, carrying that current in the field, feels a force $F = BIL$. Lenz's law tells you which way that force points.

**Predict (P1).** The rod (mass 50 g) is given a push to 2.0 m/s and released. There is no friction. Sketch a graph of its speed against time. Does it ever stop? Where does its kinetic energy go? Write your prediction down before you run the next cell.

In [ ]:
m, R, v0 = 0.050, 0.50, 2.0                # kg, ohm, m/s

def rod(t, state, B=B, L=L, m=m, R=R):   # the defaults keep this experiment's values
    x, v = state
    current = em.rod_emf(B, L, v) / R      # I = BLv / R
    return [v, -B * current * L / m]       # Lenz: the force BIL opposes the motion

tau = m * R / (B * L) ** 2                 # see the Explain answer for where this comes from
t, state = integrators.solve(rod, [0.0, v0], (0, 5 * tau), n=2001)
x, v = state.T
current = em.rod_emf(B, L, v) / R

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.plot(t, v, label="simulation")
ax1.plot(t, v0 * np.exp(-t / tau), "k:", label=r"$v_0 e^{-t/\tau}$")
ax1.set_xlabel("t / s"); ax1.set_ylabel("speed / m s$^{-1}$"); ax1.legend()
ax2.plot(t, x)
ax2.axhline(v0 * tau, color="k", ls=":", label=r"$v_0\tau$")
ax2.set_xlabel("t / s"); ax2.set_ylabel("distance travelled / m"); ax2.legend()
fig.suptitle("Rod on rails with a 0.50 ohm resistor, released at 2.0 m/s")

heat = np.trapezoid(current**2 * R, t)     # add up I^2 R dt
print(f"time constant tau = mR/(BL)^2 = {tau:.2f} s")
print(f"after 5 tau the speed is {v[-1]:.3f} m/s and the rod has gone {x[-1]:.2f} m")
print(f"kinetic energy lost: {0.5 * m * (v0**2 - v[-1]**2):.5f} J;  heat in the resistor: {heat:.5f} J")

**Explain (P1).** Why does the rod slow down? Why does it slow down more and more gently, and does it travel a finite or an infinite distance? Use the last printed line to say where the kinetic energy went.

*Try it:* double the resistance `R`. Does the rod stop sooner or later? Why?

## 4. A rotating coil: the a.c. generator

A generator turns a coil of $N$ turns and area $A$ at a steady angular speed $\omega$ in a uniform field. The angle between the field and the normal grows steadily, $\theta = \omega t$, so

$$N\Phi = BAN\cos\omega t \qquad\Rightarrow\qquad \varepsilon = -\frac{d(N\Phi)}{dt} = BAN\omega\sin\omega t .$$

The emf is **alternating**, with peak value $\varepsilon_0 = BAN\omega$. Its **r.m.s.** value, the steady voltage that would deliver the same average power to a resistor, is $\varepsilon_0/\sqrt2$.

**Predict (P2).** The coil starts with the field passing straight through it, so the flux linkage is at its maximum. As it turns through a quarter of a turn the flux linkage falls to zero. Where in that quarter turn is the emf largest: at the start, in the middle, or at the end?

In [ ]:
N, A, B, f = 100, 0.010, 0.20, 50          # turns, m^2, T, revolutions per second
omega = 2 * np.pi * f
t = np.linspace(0, 2 / f, 801)             # two turns
linkage = B * A * N * np.cos(omega * t)
emf = em.coil_emf(N, B, A, omega, t)       # BAN omega sin(omega t)
faraday = -np.gradient(linkage, t)         # minus the slope of the flux-linkage graph

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax1.plot(1e3 * t, linkage)
ax1.set_ylabel(r"$N\Phi$ / Wb")
ax2.plot(1e3 * t, emf, label=r"$BAN\omega\sin\omega t$")
ax2.plot(1e3 * t[::25], faraday[::25], "o", ms=3, label=r"$-d(N\Phi)/dt$, numerically")
ax2.set_ylabel("emf / V"); ax2.set_xlabel("t / ms"); ax2.legend(fontsize=8, loc="upper right")
for ax in (ax1, ax2):
    ax.axhline(0, color="0.7", lw=0.6)
fig.suptitle("Generator coil: 100 turns, 10 cm square, 0.20 T, 50 Hz")

for fraction, label in [(0, "start"), (1 / 8, "middle"), (1 / 4, "end")]:
    print(f"at the {label:>6} of the quarter turn: emf = {em.coil_emf(N, B, A, omega, fraction / f):6.2f} V")
print(f"peak emf BAN*omega = {B * A * N * omega:.1f} V, r.m.s. = {B * A * N * omega / np.sqrt(2):.1f} V")

**Explain (P2).** Use the slope of the flux-linkage graph to explain where the emf is zero and where it is largest.

**Predict (P3).** The same coil now turns twice as fast, at 100 revolutions per second. What happens to (a) the peak emf and (b) the frequency of the alternating emf?

In [ ]:
t = np.linspace(0, 0.04, 2001)
plt.figure(figsize=(8, 3.2))
for f in (50, 100):
    omega = 2 * np.pi * f
    emf = em.coil_emf(N, B, A, omega, t)
    plt.plot(1e3 * t, emf, label=f"{f} rev/s")
    print(f"{f:>3} rev/s: peak emf {emf.max():5.1f} V, period {1e3 / f:4.1f} ms")
plt.xlabel("t / ms"); plt.ylabel("emf / V"); plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1)); plt.title("The same coil at two speeds");

**Explain (P3).** $B$, $A$ and $N$ have not changed. Why is the peak emf different?

## 5. Induction with no moving parts

Nothing has to move. A long solenoid with $n$ turns per metre carrying a current $I$ has a uniform field $B = \mu_0 nI$ inside it (`em.solenoid_field`). Put a small *search coil* of $N_2$ turns and area $A_2$ inside, lined up with the axis. Its flux linkage is $N_2A_2\mu_0 nI$, so if the current changes,

$$\varepsilon = -N_2A_2\mu_0 n\,\frac{dI}{dt}.$$

With an alternating current $I = I_0\sin\omega t$ the emf alternates too, with amplitude $N_2A_2\mu_0 nI_0\omega$. This is the principle of the **transformer**: an alternating current in one coil induces an alternating emf in another coil that shares its flux.

Try it: measure how coil orientation affects induced emf in the [explorer](../explorer/#rp-searchcoil).

In [ ]:
n, I0, f = 2000, 3.0, 50                   # turns per metre, amps, hertz
N2, A2 = 50, 1.0e-4                        # search coil: 50 turns, 1.0 cm^2
omega = 2 * np.pi * f
t = np.linspace(0, 2 / f, 801)
current = I0 * np.sin(omega * t)
linkage = N2 * A2 * em.solenoid_field(n, current)
emf = -np.gradient(linkage, t)

fig, ax1 = plt.subplots(figsize=(8, 3.8), layout="constrained")
ax1.plot(1e3 * t, current, label="solenoid current")
ax1.set_xlabel("t / ms"); ax1.set_ylabel("current / A")
ax2 = ax1.twinx()
ax2.plot(1e3 * t, 1e3 * emf, color="C1", label="search-coil emf")
ax2.set_ylabel("emf / mV")
fig.legend(loc="outside lower center", ncol=2, fontsize=8); ax1.set_title("The emf is largest where the current changes fastest")

print(f"field amplitude in the solenoid: {em.solenoid_field(n, I0) * 1e3:.2f} mT")
print(f"emf amplitude: simulated {1e3 * np.abs(emf).max():.2f} mV, formula N2*A2*mu0*n*I0*omega = {1e3 * N2 * A2 * mu_0 * n * I0 * omega:.2f} mV")

The emf peaks when the current passes through zero, a quarter of a cycle away from the current's own peak, for the same reason as in P2.

*Try it:* change the frequency `f` to 500 Hz. The current amplitude is unchanged, so why does the emf grow? This is why transformers work only with alternating current: a steady direct current gives a steady flux and induces nothing.

**Common mistakes**

- Using the angle to the plane of the coil in $\Phi = BA\cos\theta$. The angle is measured from the **normal**.
- Mixing up flux $\Phi$ with flux linkage $N\Phi$. The emf depends on the change in **flux linkage**.
- Thinking the emf is largest when the flux is largest. It is largest when the flux is **changing fastest**.
- Quoting a peak value when the question asks for r.m.s., or the other way round: $\varepsilon_{\text{rms}} = \varepsilon_0/\sqrt2$.

---

## Exercises

Take $\mu_0 = 4\pi\times10^{-7}\ \mathrm{T\,m\,A^{-1}}$. Give answers to 3 significant figures.

**Q1.** A coil of 250 turns and area $3.0\ \mathrm{cm^2}$ sits in a uniform field of 0.12 T. The field makes an angle of $40^\circ$ with the normal to the coil. Find the flux linkage.

**Q2.** The coil in Q1 is pulled out of the field, so the flux linkage falls to zero, in 0.080 s. Find the average induced emf.

**Q3.** A rod 0.25 m long slides at a steady 3.0 m/s along rails in a 0.40 T field at right angles to the rails. A $2.0\ \Omega$ resistor joins the ends of the rails. Find the force needed to keep the rod moving at constant speed.

**Q4.** A generator coil of 150 turns and area $2.0\times10^{-3}\ \mathrm{m^2}$ turns at 50 revolutions per second in a uniform 0.20 T field. Find the r.m.s. emf.

**Q5.** A metal rod 4.0 cm long is held parallel to a long straight wire carrying 20 A and moved directly away from the wire at 3.0 m/s. Find the emf across the rod when it is 5.0 cm from the wire. (The field of the wire is $B = \mu_0 I/2\pi r$.)

**Q6 (simulation).** In section 3, how long does the rod take to slow from 2.0 m/s to 1.0 m/s? Find it from the simulation, then check it with a formula.

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_tau = 0.050 * 0.50 / (0.40 * 0.15) ** 2
_answers = {
    "Q1": 250 * 0.12 * 3.0e-4 * np.cos(np.radians(40)),
    "Q2": 250 * 0.12 * 3.0e-4 * np.cos(np.radians(40)) / 0.080,
    "Q3": (0.40 * 0.25) ** 2 * 3.0 / 2.0,
    "Q4": em.coil_emf(150, 0.20, 2.0e-3, 2 * np.pi * 50, 1 / 200) / np.sqrt(2),   # peak a quarter turn in
    "Q5": em.rod_emf(em.wire_field(20, 0.05), 0.04, 3.0),
    "Q6": _tau * np.log(2),
}

# Replace None with your answers, in SI units.
mine = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None, "Q6": None}
for q in mine:
    check(q, mine[q], _answers[q])

---

## Worked solutions

Try the questions before reading on.

**Q1.** $N\Phi = BAN\cos\theta = 0.12 \times 3.0\times10^{-4} \times 250 \times \cos 40^\circ = 6.89\times10^{-3}\ \mathrm{Wb}$ (Wb-turns).

**Q2.** $\varepsilon = \dfrac{\Delta(N\Phi)}{\Delta t} = \dfrac{6.89\times10^{-3}}{0.080} = 8.62\times10^{-2}\ \mathrm{V}$, about 86.2 mV.

**Q3.** $\varepsilon = BLv = 0.40 \times 0.25 \times 3.0 = 0.300$ V, so $I = \varepsilon/R = 0.150$ A. The force on the rod is $F = BIL = 0.40 \times 0.150 \times 0.25 = 1.50\times10^{-2}$ N, opposing the motion, so you must push with 0.0150 N. Check with energy: you supply $Fv = 0.0450$ W, and the resistor dissipates $I^2R = 0.0450$ W.

**Q4.** $\omega = 2\pi \times 50 = 314\ \mathrm{rad\,s^{-1}}$, so $\varepsilon_0 = BAN\omega = 0.20 \times 2.0\times10^{-3} \times 150 \times 314 = 18.8$ V and $\varepsilon_{\text{rms}} = \varepsilon_0/\sqrt2 = 13.3$ V.

**Q5.** At 5.0 cm, $B = \dfrac{\mu_0 I}{2\pi r} = \dfrac{4\pi\times10^{-7} \times 20}{2\pi \times 0.050} = 8.00\times10^{-5}$ T. The rod lies along the wire, so the field is the same all along it; the field, the rod and its velocity are mutually at right angles. $\varepsilon = BLv = 8.00\times10^{-5} \times 0.040 \times 3.0 = 9.60\times10^{-6}$ V, about 9.60 μV.

**Q6.** The braking force is $F = BIL = B^2L^2v/R$, so $m\,\dfrac{dv}{dt} = -\dfrac{B^2L^2}{R}v$. The deceleration is proportional to the speed, which gives exponential decay, $v = v_0e^{-t/\tau}$ with $\tau = \dfrac{mR}{B^2L^2} = 6.94$ s. The speed halves after $\tau\ln 2 = 4.81$ s. The simulation below agrees.

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:#.3g}")

_t, _state = integrators.solve(rod, [0.0, 2.0], (0, 2 * _tau), n=20001)
_half = _t[np.argmax(_state[:, 1] <= 1.0)]
print(f"simulated: the rod first drops to 1.0 m/s at t = {_half:.3f} s; tau ln 2 = {_tau * np.log(2):.3f} s")

**Next:** in section 3, give the rod a constant forward force instead of a single push, for example by adding `+ 0.010 / m` to its acceleration. What speed does it settle at, and why? (Hint: at that speed the magnetic braking force balances the push.)

### Predict and explain: answers

**P1.** The rod slows down because the induced current's force $BIL$ opposes its motion (Lenz's law). That force is proportional to the speed, $F = B^2L^2v/R$, so as the rod slows the braking gets gentler, and the speed decays exponentially, $v = v_0e^{-t/\tau}$ with $\tau = mR/B^2L^2$. In theory it never quite stops, but it travels only a finite distance, $v_0\tau$. Its kinetic energy all becomes thermal energy in the resistor: the two printed energies agree.

**P2.** At the end. The emf is minus the slope of the flux-linkage graph. At the start the flux linkage is at its maximum, the top of the cosine curve, where the graph is flat, so the emf is zero. A quarter turn later the flux linkage is zero but falling fastest, so the emf is at its peak. Flux and emf are a quarter of a cycle out of step.

**P3.** Both double. The peak emf is $\varepsilon_0 = BAN\omega$: the same change in flux linkage now happens in half the time, so its rate of change, which is the emf, is twice as large. The frequency is the number of turns per second, so it doubles from 50 Hz to 100 Hz.